In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

In [3]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")


In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

In [7]:
from pprint import pprint

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='79e6f52a-e715-4123-9cbf-6d6653b023e8'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019daba6-a236-7392-83cc-1f2911090260-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'afd52f96-eef9-4d1c-a526-18e02f78b2cf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 21, 'total_tokens': 85, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='ee1d3c18-893a-4e58-a16b-524684d9b0ad', tool_call_id='af

In [8]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='e7c46d94-53a7-453c-8549-2abd079b7833'),
              AIMessage(content="Hello! I'm doing well, thank you for asking. How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019daba8-2860-7b92-bdd0-d3900b4bef63-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 20, 'total_tokens': 85, 'input_token_details': {'cache_read': 0}})]}


## Read state

In [9]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [10]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='c106c721-9c5d-48a0-889e-35420c4ce19c'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dabaa-a54d-72f3-ad05-172477b02f2d-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': '212a0e92-e3ad-4b9f-afd2-fb3ee2f2df15', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 21, 'total_tokens': 120, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='3e2b65a7-6676-4c60-9aa1-111e92968d22', tool_call_id='2

In [11]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='c106c721-9c5d-48a0-889e-35420c4ce19c'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019dabaa-a54d-72f3-ad05-172477b02f2d-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': '212a0e92-e3ad-4b9f-afd2-fb3ee2f2df15', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 99, 'output_tokens': 21, 'total_tokens': 120, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='3e2b65a7-6676-4c60-9aa1-111e92968d22', tool_call_id='2